In [0]:
%pip install langchain==1.2.11 langchain-core==1.2.18 langchain-community==0.4.1 databricks-langchain==0.17.0 databricks-sdk==0.97.0 pandas==2.3.3 mlflow-skinny==3.10.1 mlflow[databricks]==3.10.1
%pip install langchain-experimental==0.3.4 --no-deps
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Quick sanity check
import pandas as pd
df_check = pd.read_csv("students_data.csv")
# print(f"CSV loaded OK — shape: {df_check.shape}")
# print(df_check.head(3).to_string())
df=df_check
df[(df['Subject'] == 'History') & (df['Age'] > 20)].shape[0]

3

In [0]:
# ── Cell 1: Configuration ──────────────────────────────────────────────────
from mlflow.tracking import MlflowClient

AGENT_NAME      = "customer_support_agent"
ENDPOINT_NAME   = "customer-support-endpoint"
DATABRICKS_HOST = "https://adb-7405616168141157.17.azuredatabricks.net"

# ⚠️ SECURITY: Replace hardcoded token with secrets in production
# TOKEN = dbutils.secrets.get(scope="your-scope", key="databricks-pat")
TOKEN           = "YOUR_DATABRICKS_TOKEN_HERE"

UC_MODEL_NAME   = f"nextgen_genie.default.{AGENT_NAME}"  # <catalog>.<schema>.<model>

print(f"Agent      : {AGENT_NAME}")
print(f"UC Model   : {UC_MODEL_NAME}")
print(f"Endpoint   : {ENDPOINT_NAME}")

Agent      : customer_support_agent
UC Model   : nextgen_genie.default.customer_support_agent
Endpoint   : customer-support-endpoint


In [0]:
import mlflow
import pandas as pd


class StudentsCsvAgent(mlflow.pyfunc.PythonModel):
    """
    MLflow PythonModel wrapping a LangChain Pandas DataFrame Agent.
    Every invocation is traced inside an MLflow Experiment.

    Input  schema : { "query": str }
    Output schema : { "response": str }
    """

    MAX_RETRIES = 3  # retry if the LLM fails to invoke the tool

    def load_context(self, context):
        import os, pandas as pd, mlflow
        from pydantic import BaseModel, Field, model_validator
        from databricks_langchain import ChatDatabricks
        from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

        # ── Compat shim: LLM sends {'code': ...} but PythonAstREPLTool expects {'query': ...}
        class PythonInputsCompat(BaseModel):
            query: str = Field(default="", description="code snippet to run")

            @model_validator(mode='before')
            @classmethod
            def accept_code_alias(cls, values):
                if isinstance(values, dict) and 'code' in values and 'query' not in values:
                    values['query'] = values.pop('code')
                return values

        # 1. LLM
        self.llm = ChatDatabricks(
            endpoint="databricks-gpt-oss-120b",
            max_tokens=1024,
            temperature=0.0,
        )

        # 2. Load CSV — same dir as notebook
        csv_path = os.getenv("STUDENTS_CSV_PATH", "students_data.csv")
        self.df = pd.read_csv(csv_path)
        num_rows = len(self.df)
        col_info = ", ".join(f"{c} ({self.df[c].dtype})" for c in self.df.columns)

        # 3. Build Pandas Agent
        self.agent = create_pandas_dataframe_agent(
            llm=self.llm,
            df=self.df,
            agent_type="tool-calling",
            verbose=True,
            allow_dangerous_code=True,
            number_of_head_rows=0,  # prevent LLM from recreating df from sample rows
            prefix=(
                "You are a data analyst assistant. "
                f"The variable `df` is ALREADY loaded in the Python environment with ALL {num_rows} rows. "
                f"Columns: {col_info}. "
                "CRITICAL RULES: "
                "1. NEVER create a new DataFrame. NEVER reassign `df`. Just use `df` directly. "
                "2. The Sports column contains comma-separated values "
                "(e.g. 'Cricket, Swimming'). When filtering by a sport, "
                "always use df['Sports'].str.contains('SportName', na=False). "
                "3. ALWAYS invoke the python_repl_ast tool to run code before answering. "
                "4. Report the EXACT numeric result returned by the tool. "
                "5. Respond with ONLY a plain-text answer. No JSON. No reasoning blocks."
            ),
        )

        # 4. Patch tool schema so 'code' is accepted as alias for 'query'
        for tool in self.agent.tools:
            if tool.name == "python_repl_ast":
                tool.args_schema = PythonInputsCompat

        # 5. Set MLflow Experiment for tracing
        mlflow.set_experiment("/Shared/students_csv_agent_experiment")
        print("StudentsCsvAgent loaded ✅")

    # ── helpers ────────────────────────────────────────────────────────────

    @staticmethod
    def _extract_text(raw_output: str) -> str | None:
        """Return clean text from structured JSON, or None if only reasoning."""
        import json
        try:
            blocks = json.loads(raw_output)
            if not isinstance(blocks, list):
                return raw_output
        except (json.JSONDecodeError, TypeError):
            return raw_output          # already plain text

        # 1. Prefer explicit "text" type blocks
        texts = [b["text"] for b in blocks
                 if isinstance(b, dict) and b.get("type") == "text" and b.get("text")]
        if texts:
            return "\n".join(texts)

        # 2. Only reasoning — signal caller to retry
        return None

    def _invoke_with_retry(self, query: str) -> str:
        """Invoke the agent up to MAX_RETRIES times until we get a real answer."""
        for attempt in range(1, self.MAX_RETRIES + 1):
            result = self.agent.invoke({"input": query})
            raw = result.get("output", str(result))
            clean = self._extract_text(raw)
            if clean is not None:          # got a real answer
                return clean
            print(f"  ⚠️  Attempt {attempt}/{self.MAX_RETRIES}: LLM returned reasoning only — retrying...")
        # exhausted retries — return best-effort from last raw output
        return raw                          # noqa: F821 — raw is set in last iteration

    # ── predict ────────────────────────────────────────────────────────────

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        import mlflow

        # Extract query
        query = None
        if isinstance(model_input, pd.DataFrame):
            query = model_input["query"].iloc[0] if "query" in model_input.columns else str(model_input.iloc[0, 0])
        elif isinstance(model_input, dict):
            query = model_input.get("query") or str(next(iter(model_input.values())))
        elif isinstance(model_input, list):
            first = model_input[0]
            query = first.get("query") if isinstance(first, dict) else str(first)
        else:
            query = str(model_input)

        # Run agent inside an MLflow Experiment run
        with mlflow.start_run(run_name="csv_agent_query", nested=True):
            mlflow.log_param("query", query[:250])
            try:
                answer = self._invoke_with_retry(query)
            except Exception as exc:
                answer = f"Agent error: {exc}"
            mlflow.log_metric("success", 1 if not answer.startswith("Agent error") else 0)
            mlflow.log_text(answer, "agent_response.txt")

        return pd.DataFrame([{"response": answer}])


print("StudentsCsvAgent class defined ✅")

StudentsCsvAgent class defined ✅


In [0]:
import mlflow, pandas as pd, glob
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import ColSpec, Schema
CSV_UC_PATH   = "students_data.csv"
input_schema  = Schema([ColSpec("string", "query")])
output_schema = Schema([ColSpec("string", "response")])
signature     = ModelSignature(inputs=input_schema, outputs=output_schema)


pip_reqs = [
    "mlflow==3.10.1",
    "langchain==1.2.11",
    "langchain-core==1.2.18",
    "langchain-community==0.4.1",
    "langchain-experimental==0.3.4",
    "databricks-langchain==0.17.0",
    "databricks-sdk==0.97.0",
    "pandas==2.3.3",
]

input_example = pd.DataFrame([{"query": "How many students play Cricket?"}])
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="register-students-csv-agent") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model=StudentsCsvAgent(),
        signature=signature,
        pip_requirements=pip_reqs,
        registered_model_name=UC_MODEL_NAME,
        input_example=input_example,
        model_config={"STUDENTS_CSV_PATH": CSV_UC_PATH},
    )
    print(f"Run ID        : {run.info.run_id}")
    print(f"Model URI     : {model_info.model_uri}")
    print(f"Registered as : {UC_MODEL_NAME}")

client = MlflowClient()
latest_version = max(
    int(mv.version)
    for mv in client.search_model_versions(f"name='{UC_MODEL_NAME}'")
)
print(f"Latest version: {latest_version}")

🔗 View Logged Model at: https://adb-7405616168141157.17.azuredatabricks.net/ml/experiments/403923384854529/models/m-05d304adfdcf4bda9c1fbf3eac40cb9c?o=7405616168141157
2026/03/15 01:30:40 INFO mlflow.pyfunc: Validating input example against model signature
2026/03/15 01:30:42 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "q.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: cannot import name 'AgentType' from 'langchain.agents' (/local_disk0/.ephemeral_nfs/envs/pythonEnv-bdf1d985-062f-44c3-9d11-ede9abb189ea/lib/python3.10/site-packages/langchain/agents/__i

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '44' of model 'nextgen_genie.default.customer_support_agent': https://adb-7405616168141157.17.azuredatabricks.net/explore/data/models/nextgen_genie/default/customer_support_agent/version/44?o=7405616168141157


Run ID        : 9e7bfee23fce403fa899ab2b66faf3e7
Model URI     : models:/m-05d304adfdcf4bda9c1fbf3eac40cb9c
Registered as : nextgen_genie.default.customer_support_agent
Latest version: 44


In [0]:
# Compatibility shim: langchain 1.x moved most submodules to langchain_classic.
# langchain-experimental 0.3.4 still imports from langchain.* — redirect as fallback.
import sys, importlib

class _LCClassicFallback:
    def find_module(self, fullname, path=None):
        if fullname.startswith('langchain.') and fullname not in sys.modules:
            classic = fullname.replace('langchain.', 'langchain_classic.', 1)
            try:
                importlib.import_module(classic)
                return self
            except ImportError:
                return None
        return None

    def load_module(self, fullname):
        if fullname in sys.modules:
            return sys.modules[fullname]
        classic = fullname.replace('langchain.', 'langchain_classic.', 1)
        mod = importlib.import_module(classic)
        sys.modules[fullname] = mod
        return mod

sys.meta_path.insert(0, _LCClassicFallback())

import langchain_classic.agents as _ca
import langchain.agents as _na
for _n in dir(_ca):
    if not _n.startswith('_') and not hasattr(_na, _n):
        setattr(_na, _n, getattr(_ca, _n))

# ── Original cell logic ──
from mlflow.models import validate_serving_input, convert_input_example_to_serving_input

model_uri     = f"models:/{UC_MODEL_NAME}/{latest_version}"
serving_input = convert_input_example_to_serving_input(
    pd.DataFrame([{"query": "How many students play Cricket?"}])
)
print("Serving payload:")
print(serving_input)

print("\nValidating model locally...")
try:
    validate_serving_input(model_uri, serving_input)
    print("\n✅ Validation passed — safe to deploy")
except Exception as e:
    print(f"\n❌ Validation FAILED — do NOT deploy\n   Error: {e}")
    raise

Serving payload:
{
  "dataframe_split": {
    "columns": [
      "query"
    ],
    "data": [
      [
        "How many students play Cricket?"
      ]
    ]
  }
}

Validating model locally...


StudentsCsvAgent loaded ✅


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'code': "df['Sports'].str.contains('Cricket', na=False).sum()"}`


5050

> Finished chain.

✅ Validation passed — safe to deploy


In [0]:
from collections import Counter

loaded_model = mlflow.pyfunc.load_model(model_uri)
# test_input = pd.DataFrame([{"query": "How many history student are of age greater than 20?"}])
# test_input = pd.DataFrame([{"query": "what is feedback for deepak verma?"}])
# test_input = pd.DataFrame([{"query": "what is feedback for deepak verma?"}])
# test_input = pd.DataFrame([{"query": "what is feedback for shreya ?"}])
# test_input = pd.DataFrame([{"query": "what are the three names with subject geography ?"}])
# test_input = pd.DataFrame([{"query": "how many students with a+ grade and chemistry subject?"}])
test_input = pd.DataFrame([{"query": "what is feedback for deepak verma (19)?"}])


NUM_TRIALS = 3
responses = []

for i in range(1, NUM_TRIALS + 1):
    print(f"\n{'='*60}")
    print(f"  Trial {i}/{NUM_TRIALS}")
    print(f"{'='*60}")
    try:
        result = loaded_model.predict(test_input)
        answer = result["response"].iloc[0]
        responses.append(answer)
        print(f"  Response: {answer}")
    except Exception as e:
        responses.append(f"ERROR: {e}")
        print(f"  ERROR: {e}")

print(f"\n{'='*60}")
print(f"  Summary — {NUM_TRIALS} trials")
print(f"{'='*60}")
for i, r in enumerate(responses, 1):
    status = "✅" if not r.startswith("ERROR") and not r.startswith("Agent error") else "❌"
    print(f"  Trial {i} {status}: {r[:120]}{'...' if len(r) > 120 else ''}")

successes = [r for r in responses if not r.startswith("ERROR") and not r.startswith("Agent error")]
print(f"\n  Success rate: {len(successes)}/{NUM_TRIALS}")

StudentsCsvAgent loaded ✅

  Trial 1/3


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'code': "df.loc[(df['Name']=='Deepak Verma') & (df['Age']==19), 'Teacher Feedback']"}`


2    A dedicated student who consistently demonstra...
Name: Teacher Feedback, dtype: objectA dedicated student who consistently demonstrates a strong work ethic and a keen interest in learning.

> Finished chain.
  Response: A dedicated student who consistently demonstrates a strong work ethic and a keen interest in learning.

  Trial 2/3


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'code': "df.loc[(df['Name']=='Deepak Verma') & (df['Age']==19), 'Teacher Feedback']"}`


2    A dedicated student who consistently demonstra...
Name: Teacher Feedback, dtype: objectA dedicated student who consistently demonstrates a strong work ethic and a keen interest in learning.

> Finished chain.
  Response: A dedicated student who consistently demonstrates a strong work